In [4]:
print('hello')

hello


## Setup

Initialize Earth Engine

Build AOI and Collection

In [5]:
import ee
from pathlib import Path
from dotenv import load_dotenv
from rs_timeseries.config import load_config, get_ee_project
from rs_timeseries.io import build_aoi, build_merged_collection

# Resolve the project root regardless of where the notebook is located
PROJECT_ROOT = Path("D:/Projects/landsat-timeseries")

load_dotenv(PROJECT_ROOT / ".env")
project = get_ee_project()
print(f"Using project: {project}")

ee.Initialize(project=project)

config = load_config(PROJECT_ROOT / "configs/swiss_alps_lst.yaml")
aoi = build_aoi(config)
collection = build_merged_collection(config, aoi)

n = collection.size().getInfo()
print(f"Collection size: {n} images")

Using project: goekdeniz
Collection size: 16088 images


## Preprocessing
Apply Scaling and Cloud Masking
Verify Band Names
Verify LST Values at Sample Points
Verify Median LST Across Time Series

In [6]:
from rs_timeseries.preprocessing import preprocess_collection

preprocessed = preprocess_collection(collection, config)
preprocessed = preprocessed.select("SurfT")

In [ ]:
from rs_timeseries.preprocessing import preprocess_collection

preprocessed = preprocess_collection(collection, config)
preprocessed = preprocessed.select("SurfT")

# Check scene count survived masking
n_pre = preprocessed.size().getInfo()
print(f"Scenes after preprocessing: {n_pre}")

# Inspect one image — should only have SurfT and Emis bands
sample = preprocessed.first()
print("Bands after masking:", sample.bandNames().getInfo())
# Expected: ['SurfT', 'Emis']

# Check a pixel value at a known location in the Swiss Alps
point = ee.Geometry.Point([8.0, 46.5])
value = sample.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point,
    scale=30,
).getInfo()
print("Sample LST value (Kelvin):", value)
# Expected: a value between ~213 K (−60°C) and ~333 K (60°C)


In [ ]:
# Try several points across the Alps
test_points = {
    "Zurich":  ee.Geometry.Point([8.55, 47.37]),
    "Geneva":  ee.Geometry.Point([6.15, 46.20]),
    "Innsbruck": ee.Geometry.Point([11.39, 47.26]),
}

for name, point in test_points.items():
    val = preprocessed.first().reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=30,
    ).getInfo()
    print(f"{name}: {val}")

In [ ]:
median_lst = preprocessed.select("SurfT").median()
val = median_lst.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=ee.Geometry.Point([8.0, 46.5]),
    scale=30,
).getInfo()
print(f"Median LST at test point: {val}")
# Expected: ~280 K

## Harmonic Modeling


In [ ]:
from rs_timeseries.modeling import (
    add_harmonic_predictors,
    fit_harmonic_regression,
    add_fitted_values,
)

# Add predictor bands to the collection
collection_with_predictors = preprocessed.map(
    lambda img: add_harmonic_predictors(img, config)
)

# Fit the regression — this is a GEE server-side operation,
# nothing runs locally until we call .getInfo()
coefficients = fit_harmonic_regression(collection_with_predictors, config)

# Inspect coefficient values at a point to sanity check the model
point = ee.Geometry.Point([8.0, 46.5])
coeff_values = coefficients.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point,
    scale=30,
).getInfo()
print("Coefficients at sample point:")
print(f"  b0 (MALST, K):    {coeff_values.get('b0'):.2f}")
print(f"  b1 (trend, K/yr): {coeff_values.get('b1'):.4f}")
print(f"  b2 (cos):         {coeff_values.get('b2'):.2f}")
print(f"  b3 (sin):         {coeff_values.get('b3'):.2f}")

# Expected:
# b0 ~ 280 K (typical mean LST in the Alps)
# b1 ~ small positive number (e.g. 0.03 K/yr warming trend)
# b2, b3 ~ values related to the seasonal amplitude


In [ ]:
from rs_timeseries.products import compute_std, compute_variance

std_img = compute_std(preprocessed, "SurfT")
var_img = compute_variance(preprocessed, "SurfT")

point = ee.Geometry.Point([8.55, 47.37])  # Zurich

std_val = std_img.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point,
    scale=30,
).getInfo()

var_val = var_img.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point,
    scale=30,
).getInfo()

print(f"Std  (stored): {std_val}")   # divide by 100 to get K
print(f"Var  (stored): {var_val}")   # divide by 10 to get K²

In [ ]:
from rs_timeseries.extraction import extract_point_timeseries

df = extract_point_timeseries(
    collection  = preprocessed,
    lon         = 8.55,
    lat         = 47.37,
    band        = "SurfT",
    scale       = 30,
    start_year  = 1984,
    end_year    = 2024,
    verbose     = True,
)

df.to_csv("notebooks/timeseries_zurich.csv", index=False)

In [ ]:
# Cell: plot raw observations + harmonic fit
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(14, 4))
ax.scatter(df["date"], df["value"]-273.15, s=2, alpha=0.4, color="steelblue", label="Observed LST")
ax.set_ylabel("LST (Kelvin)")
ax.set_title("Landsat LST — Zurich [8.55, 47.37]  |  1984–2024")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Verify the cloud band is present
sample = preprocessed.first().bandNames().getInfo()
print(sample)
# Expected: ['SurfT', 'Emis', 'cloud', ...]

# Extract with cloud flag at one point
from rs_timeseries.extraction import extract_point_timeseries

df = extract_point_timeseries(
    collection  = preprocessed,
    lon         = 8.55,
    lat         = 47.37,
    bands       = ["SurfT", "cloud"],
    scale       = 30,
    start_year  = 2020,   # small range first
    end_year    = 2022,
    verbose     = True,
)

print(df.head(10))
print(f"\nCloud coverage: {df['cloud'].mean():.1%}")